# exp_08c Outlier Ablation

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

In [ ]:
# Load prior decisions from exp_08 and exp_08b
config = load_experiment_config()
setup_path = OUTPUT_DIR / 'metadata/setup_decisions.json'

if not setup_path.exists():
    raise FileNotFoundError(f"setup_decisions.json not found. Run exp_08 and exp_08b first.")

setup = json.loads(setup_path.read_text())

if 'chm_strategy' not in setup:
    raise ValueError("chm_strategy not found. Run exp_08 first.")
if 'proximity_strategy' not in setup:
    raise ValueError("proximity_strategy not found. Run exp_08b first.")

chm_strategy = setup['chm_strategy']['decision']
chm_features = setup['chm_strategy'].get('included_features', [])
proximity_strategy = setup['proximity_strategy']['decision']

print(f"Loaded CHM decision: {chm_strategy}")
print(f"Loaded proximity decision: {proximity_strategy}")
print(f"CHM features: {chm_features or 'none (Sentinel-2 only)'}")

In [ ]:
# Test outlier removal variants with CHM + proximity decisions applied
variants = config['setup_ablation']['outliers']['variants']

# Load correct proximity variant
train_df, val_df, _ = data_loading.load_berlin_splits(
    DATA_DIR / 'phase_2_splits', variant=proximity_strategy
)

# Apply CHM strategy
train_df = ablation.apply_chm_strategy(train_df, chm_strategy)
val_df = ablation.apply_chm_strategy(val_df, chm_strategy)

results = []
for variant in variants:
    df_train = train_df.copy()
    df_val = val_df.copy()

    # Apply outlier removal
    df_train = ablation.apply_outlier_removal(df_train, variant['name'])
    df_val = ablation.apply_outlier_removal(df_val, variant['name'])

    # Get feature columns
    feature_cols = data_loading.get_feature_columns(
        df_train,
        include_chm=bool(chm_features),
        chm_features=chm_features or None,
    )

    x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(
        df_train[feature_cols], x_val=df_val[feature_cols]
    )

    y_train, label_to_idx, _ = preprocessing.encode_genus_labels(df_train['genus_latin'])
    y_val = df_val['genus_latin'].map(label_to_idx).to_numpy()

    cv = training.create_spatial_block_cv(df_train, n_splits=config['global']['cv_folds'])

    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(
        rf, x_train_scaled, y_train, df_train['block_id'].values, cv
    )

    results.append({
        'variant': variant['name'],
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_val_gap': metrics['train_val_gap'],
        'samples_retained': len(df_train),
    })

results_df = pd.DataFrame(results)
fig_dir = OUTPUT_DIR / 'figures/exp_08c_outlier'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_ablation_comparison(
    results_df,
    title='Outlier Removal Comparison',
    output_path=fig_dir / 'outlier_ablation.png',
)

print('\nOutlier ablation results:')
print(results_df.to_string(index=False))

In [ ]:
# Apply decision rules
rules = config['setup_ablation']['outliers']['decision_rules']

no_removal_row = results_df.loc[results_df['variant'] == 'no_removal'].iloc[0]
baseline_n = int(no_removal_row['samples_retained'])
baseline_f1 = no_removal_row['val_f1_mean']

# Evaluate removal strategies
best_variant = 'no_removal'
best_f1 = baseline_f1
reasoning = 'Baseline (no removal) retained by default'

for variant_name in ['remove_high', 'remove_high_medium']:
    row = results_df.loc[results_df['variant'] == variant_name].iloc[0]
    variant_n = int(row['samples_retained'])
    variant_f1 = row['val_f1_mean']
    sample_loss = 1 - (variant_n / baseline_n)
    f1_improvement = variant_f1 - baseline_f1

    # Check if removal strategy passes thresholds
    if sample_loss <= rules['max_sample_loss'] and f1_improvement >= rules['min_improvement']:
        if variant_f1 > best_f1:
            best_variant = variant_name
            best_f1 = variant_f1
            reasoning = f"{variant_name}: {f1_improvement:+.3f} F1 with {sample_loss:.1%} sample loss"

decision = best_variant
final_n = int(results_df.loc[results_df['variant'] == decision, 'samples_retained'].iloc[0])
final_sample_loss = 1 - (final_n / baseline_n)

print(f"\nOutlier decision: {decision}")
print(f"Reasoning: {reasoning}")
print(f"Sample loss: {final_sample_loss:.1%} ({baseline_n} → {final_n})")
print(f"F1: {best_f1:.4f}")

In [ ]:
# Update setup_decisions.json
setup['outlier_strategy'] = {
    'decision': decision,
    'reasoning': reasoning,
    'sample_counts': {
        'baseline_n': baseline_n,
        'filtered_n': final_n,
        'sample_loss_pct': float(final_sample_loss),
    },
    'ablation_results': results_df.to_dict(orient='records'),
}

setup_path.write_text(json.dumps(setup, indent=2))
print(f"\nUpdated {setup_path} with outlier decision")
print("✅ Outlier decision saved successfully")
print(f"\n⚠️  Note: Full schema validation will run after exp_09")
print(f"   (when all strategies are complete: CHM, Proximity, Outlier, Features)")